In [ ]:
!pip install torch==2.4.0 vllm==0.6.3 xformers==0.0.27.post2 optimum[onnxruntime] python-docx qdrant-client FlagEmbedding

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of optimum-onnx[onnxruntime] to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 614.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s 

In [ ]:
!pip install "diffusers<0.35.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 38.5 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.38.0
    Uninstalling diffusers-0.38.0:
      Successfully uninstalled diffusers-0.38.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lm-format-enforcer

In [ ]:
!pip install "transformers==4.47.1" --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 121.0 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [ ]:
!pip install json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y torchcodec

Found existing installation: torchcodec 0.11.0+cu128
Uninstalling torchcodec-0.11.0+cu128:
  Successfully uninstalled torchcodec-0.11.0+cu128


In [ ]:
!pip install torch==2.4.0

In [ ]:
import os, gc, json, ast, math, hashlib
import pandas as pd
import numpy as np
import torch
from datetime import datetime
from tqdm import tqdm
from qdrant_client import QdrantClient
from qdrant_client.models import HnswConfigDiff
from qdrant_client.models import (
    PointStruct, VectorParams, Distance, PointIdsList
)
from huggingface_hub import snapshot_download
import json_repair

try:
    from dateutil import parser as _dp
    def _parse_date(s):
        try: return _dp.parse(str(s)) if s and not pd.isna(s) else None
        except: return None
except ImportError:
    def _parse_date(s): return None

from vllm import LLM, SamplingParams
from sentence_transformers import SentenceTransformer, CrossEncoder  # BUG 1 FIX: added CrossEncoder

# ══════════════════════════════════════════════════════════
# ⚙️ CONFIGURATION & TEST TOGGLE
# ══════════════════════════════════════════════════════════
IS_TEST_RUN = False
TEST_LIMIT  = 1000

DRIVE_DIR          = "/content/drive/MyDrive/redrob_ai"
DATA_PATH          = os.path.join(DRIVE_DIR, "candidates_full_dataset.csv")
JD_PATH            = os.path.join(DRIVE_DIR, "job_description.docx")

MODELS_DIR         = os.path.join(DRIVE_DIR, "models")
QWEN_7B_PATH       = os.path.join(MODELS_DIR, "Qwen2.5-7B-Instruct-AWQ")
QWEN_3B_PATH       = os.path.join(MODELS_DIR, "Qwen2.5-3B-Instruct-AWQ")
BGE_M3_PATH        = os.path.join(MODELS_DIR, "bge-m3")

if IS_TEST_RUN:
    print(f"⚠️ RUNNING IN TEST MODE ({TEST_LIMIT} Candidates)")
    PHASE1A_CSV  = os.path.join(DRIVE_DIR, "TEST_v13_phase1a.csv")
    PHASE1B_CSV  = os.path.join(DRIVE_DIR, "TEST_v8.2.2_phase1b.csv")
    QDRANT_PATH  = os.path.join(DRIVE_DIR, "TEST_v8_qdrant")
    FINAL_CSV    = os.path.join(DRIVE_DIR, "TEST_v8_submission.csv")
    TARGET_ROWS  = TEST_LIMIT
else:
    print("🔥 RUNNING IN PRODUCTION MODE (100,000 Candidates)")
    PHASE1A_CSV  = os.path.join(DRIVE_DIR, "prod_v004_phase1a.csv")
    PHASE1B_CSV  = os.path.join(DRIVE_DIR, "prod_v004_phase1b.csv")
    QDRANT_PATH  = "/content/qdrant_storage"
    FINAL_CSV    = os.path.join(DRIVE_DIR, "team_cuda,coffee,code_submission004.csv")
    TARGET_ROWS  = 100_000

CHUNK_SIZE     = 2_500
TOP_QDRANT_N   = 300
RERANK_N       = 150
SUBMISSION_N   = 100
HP_THRESHOLD   = 0.80

QUANT_3B = "awq"
QUANT_7B = "awq"

# ══════════════════════════════════════════════════════════
# SCHEMA & KEYWORDS
# ══════════════════════════════════════════════════════════
JD_CORE_SKILLS = {
    "embedding", "vector database", "hybrid search", "qdrant", "faiss", "milvus",
    "retrieval", "reranker", "bm25", "semantic search", "rag", "fine-tuning",
    "evaluation", "ndcg", "mrr", "map", "python", "production ml", "ranking",
    "information retrieval", "dense retrieval", "sparse retrieval", "sentence transformers",
    "cross-encoder", "bi-encoder", "ann", "hnsw", "pgvector",
}

DEPLOYMENT_KEYWORDS = {
    "served", "deployed", "production", "api", "inference", "throughput",
    "latency", "pipeline", "a/b test", "scaled", "realtime", "sla", "qps",
    "release", "shipped", "live", "requests per second", "endpoint",
    "vercel", "render", "railway", "netlify", "supabase", "ci/cd",
    "webhooks", "serverless", "cloud", "hosting", "containerized",
    "docker", "infrastructure", "latency-optimized", "websockets",
    "environment", "provisioned", "uptime", "build", "deployment-failure",
}

FRAMEWORK_WRAPPER_TERMS = {
    "langchain", "crewai", "autogpt", "dspy", "llamaindex",
    "flowise", "langflow", "agentgpt", "babyagi",
    "langgraph", "haystack", "semantic-kernel", "mastra",
    "ag2", "camel", "swarm", "metagpt",
    "pydantic-ai", "smolagents", "adk",
}

EXTRACTION_SCHEMA_STRING = """{
    "ai_skills": ["array of strings"],
    "deployment_evidence": boolean,
    "eval_metrics_exp": boolean,
    "scale_evidence": boolean,
    "domain": "nlp_ir" OR "cv_robotics" OR "general_ml" OR "data_eng" OR "software_only" OR "other",
    "product_co_evidence": boolean,
    "implicit_signal": "string",
    "implicit_signal": "string",
    "coding_recency_years": integer
}"""

ASPECT_WEIGHTS = {"skills": 0.50, "career": 0.40, "education": 0.10}

# ══════════════════════════════════════════════════════════
# UTILITIES & SETUP
# ══════════════════════════════════════════════════════════
def flush():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
        destroy_model_parallel()
        destroy_distributed_environment()
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()

def setup_local_models():
    print("=" * 55 + "\n📥 CHECKING LOCAL MODEL STORAGE\n" + "=" * 55)
    os.makedirs(MODELS_DIR, exist_ok=True)
    models_to_check = {
        "Qwen/Qwen2.5-7B-Instruct-AWQ": QWEN_7B_PATH,
        "Qwen/Qwen2.5-3B-Instruct-AWQ": QWEN_3B_PATH,
        "BAAI/bge-m3": BGE_M3_PATH,
    }
    for repo_id, local_path in models_to_check.items():
        if not os.path.exists(local_path) or not os.listdir(local_path):
            snapshot_download(repo_id=repo_id, local_dir=local_path)
    print("✅ Models ready.\n")

def safe_eval(val):
    if pd.isna(val) if not isinstance(val, (dict, list)) else False: return {}
    if isinstance(val, (dict, list)): return val
    try: return ast.literal_eval(str(val).strip())
    except: return {}

def safe_list(val):
    r = safe_eval(val)
    return r if isinstance(r, list) else []

def cid_to_point_id(cid) -> int:
    return int(hashlib.md5(str(cid).encode()).hexdigest()[:15], 16) % (2 ** 53)

def tier_from_score(score: float) -> str:
    """
    Calibrated for the corrected score range.
    With the double-multiplier removed, expect max ~0.55–0.65.
    T1 = top ~5%  (≥0.52)
    T2 = top ~20% (≥0.44)
    T3 = top ~50% (≥0.38)
    T4 = top ~80% (≥0.30)
    T5 = rest
    """
    if score >= 0.60: return "T1"
    if score >= 0.50: return "T2"
    if score >= 0.40: return "T3"
    if score >= 0.30: return "T4"
    return "T5"

def clean_json_output(raw_text: str) -> str:
    raw = raw_text.strip()
    if raw.startswith("```json"): raw = raw[7:]
    if raw.startswith("```"):     raw = raw[3:]
    if raw.endswith("```"):       raw = raw[:-3]
    return raw.strip()

# ══════════════════════════════════════════════════════════
# INTELLIGENCE ENGINE
# ══════════════════════════════════════════════════════════
def compute_honeypot_risk(career: list, profile: dict, skills: list) -> tuple:
    risk  = 0.0
    flags = []
    tenure_months = 0
    spans = []
    for r in career:
        if not isinstance(r, dict): continue
        s = _parse_date(r.get("start_date"))
        e = _parse_date(r.get("end_date")) or datetime.now()
        if s and e > s:
            tenure_months += (e.year - s.year) * 12 + (e.month - s.month)
            spans.append((s, e))

    claimed_months = float(profile.get("years_of_experience", 0) or 0) * 12
    if claimed_months > 0 and abs(tenure_months - claimed_months) > 30:
        risk += 0.40; flags.append("tenure_yoe_mismatch")

    if len(spans) >= 2:
        cal_months = ((max(e for _, e in spans) - min(s for s, _ in spans)).days / 30) or 1
        if tenure_months / cal_months > 1.15:
            risk += 0.30; flags.append("concurrent_role_overlap")

    for sk in skills:
        if not isinstance(sk, dict): continue
        prof = str(sk.get("proficiency", "")).lower()
        asmt = float(sk.get("assessment_score", 50) or 50)
        if prof in ("beginner", "novice") and asmt > 85:
            risk += 0.10; flags.append("assessment_proficiency_mismatch"); break
        if prof == "expert" and asmt < 25:
            risk += 0.20; flags.append("assessment_proficiency_mismatch"); break

    return min(risk, 1.0), flags

def compute_behavioral_composite(sigs: dict, profile: dict) -> tuple:
    penalties = 0.0
    boosts    = 0.0
    flags     = []

    notice = int(sigs.get("notice_period_days", 60) or 60)
    notice_m = (1.00 if notice <= 30 else 0.97 if notice <= 60
                else 0.95 if notice <= 90 else 0.90 if notice <= 120 else 0.55)

    last_active = _parse_date(sigs.get("last_active_date") or sigs.get("last_active"))
    if last_active:
        inactive = (datetime.now() - last_active).days
        if   inactive > 540: penalties += 0.45; flags.append("inactive_18m+")
        elif inactive > 180: penalties += 0.32; flags.append("inactive_6m+")
        elif inactive > 90:  penalties += 0.5

    resp = float(sigs.get("recruiter_response_rate", -1) or -1)
    if   resp > 0.70:       boosts    += 0.08
    elif 0 < resp < 0.20:   penalties += 0.05; flags.append("low_response_rate")

    accept = float(sigs.get("offer_acceptance_rate", -1) or -1)
    if   accept > 0.80:     boosts    += 0.03
    elif 0 < accept < 0.20: penalties += 0.02; flags.append("low_offer_acceptance")

    saved = int(sigs.get("saved_by_recruiters_30d", 0) or 0)
    if   saved >= 5: boosts += 0.08
    elif saved >= 2: boosts += 0.03

    github = float(sigs.get("github_activity_score", -1) or -1)
    g_norm = (github / 10.0) if github >= 0 else 0.0
    if   g_norm > 0.70:                     boosts    += 0.15
    elif github < 0 or g_norm < 0.10:       penalties += 0.03; flags.append("no_public_code")

    li_conn      = int(sigs.get("linkedin_connections", 0) or 0)
    endorsements = int(sigs.get("endorsements_count",   0) or 0)
    ext_val = (0.6 * g_norm
             + 0.3 * min(li_conn / 500, 1.0)
             + 0.1 * min(endorsements / 50, 1.0))
    if ext_val < 0.10: penalties += 0.025

    sal_str = str(sigs.get("expected_salary_range_inr_lpa", "") or "")
    try:
        nums = [float(x) for x in sal_str.replace(" ", "").split("-")
                if x.replace(".", "").isdigit()]
        if nums and max(nums) > 120:
            penalties += 0.08; flags.append("salary_mismatch")
    except Exception:
        pass

    completeness = float(sigs.get("profile_completeness", 0.5) or 0.5)
    if completeness < 0.4: penalties += 0.03

    loc      = str(profile.get("location", sigs.get("location", "")) or "").lower()
    relocate = str(sigs.get("willing_to_relocate", "false")).lower() in ("true", "yes", "1")
    mode     = str(sigs.get("preferred_work_mode", "") or "").lower()

    if   any(c in loc for c in ["pune", "noida"]):
        loc_m = 1.00
    elif any(c in loc for c in ["hyderabad", "mumbai", "delhi", "bengaluru", "bangalore", "ncr"]):
        loc_m = 0.95
    elif "india" in loc:
        loc_m = 0.95 if relocate else 0.92
    elif relocate:
        loc_m = 0.90
    else:
        loc_m = 0.90; flags.append("outside_india_no_relocation")

    if "remote" in mode and "hybrid" not in mode:
        loc_m = min(loc_m, 0.90)

    behavioral_score = min(max(0.50 + boosts - penalties, 0.05), 1.0)
    return behavioral_score, loc_m, notice_m, flags, notice

def compute_rule_signals(career: list) -> tuple:
    penalties = 0.0
    boosts    = 0.0
    flags     = []

    if not career: return penalties, boosts, flags

    desc = " ".join(str(r.get("description", "") if isinstance(r, dict) else "").lower()
                    for r in career)
    role_titles = [str(r.get("title", "") if isinstance(r, dict) else "").lower()
                   for r in career]

    has_wrapper = any(t in desc for t in FRAMEWORK_WRAPPER_TERMS)
    has_pre_llm = any(str(y) in desc for y in range(2010, 2022))
    if has_wrapper and not has_pre_llm:
        penalties += 0.15; flags.append("shallow_ai_era")
    elif any(str(y) in desc for y in range(2015, 2020)) and "nlp" in desc:
        boosts += 0.08

    research_role_count = sum(
        1 for t in role_titles
        if any(term in t for term in {"researcher", "scientist", "postdoc", "fellow", "phd intern"})
    )
    if len(career) > 0 and (research_role_count / len(career)) > 0.70:
        penalties += 0.35; flags.append("research_only")

    if not any(kw in desc for kw in DEPLOYMENT_KEYWORDS) and len(career) >= 4:
        penalties += 0.05; flags.append("no_deployment_evidence")

    if role_titles:
        latest = role_titles[-1]
        if any(kw in latest for kw in ["director", "vp ", "head of", "chief"]) and "engineer" not in latest:
            penalties += 0.08; flags.append("mgmt_only_recent")

    return penalties, boosts, flags

def score_from_extraction(raw_json: str) -> tuple:
    flags = []
    try:
        ext = json_repair.loads(raw_json)
        if not isinstance(ext, dict): ext = {}
    except Exception:
        return 0.15, ["extraction_parse_failed"]

    score     = 0.0
    ai_skills = {s.lower() for s in ext.get("ai_skills", [])}
    overlap   = len(ai_skills & JD_CORE_SKILLS)
    score += min(overlap / max(len(JD_CORE_SKILLS) * 0.25, 1), 1.0) * 0.40

    if ext.get("deployment_evidence"): score += 0.20


    if ext.get("eval_metrics_exp"): score += 0.10
    if ext.get("scale_evidence"):   score += 0.05

    domain_map = {
        "nlp_ir": 1.0, "general_ml": 0.25, "data_eng": 0.30,
        "software_only": 0.10, "cv_robotics": 0.10, "other": 0.08,
    }
    score += domain_map.get(ext.get("domain", "other"), 0.10) * 0.30

    if ext.get("product_co_evidence"): score += 0.10

    recency = int(ext.get("coding_recency_years", 0) or 0)
    if   recency > 18: score += 0.10; flags.append(f"inactive_{recency}yr_coding")
    elif recency > 12: score -= 0.08

    return max(0.0, min(score, 1.0)), flags

# ══════════════════════════════════════════════════════════
# PHASE 0: JD INTELLIGENCE
# ══════════════════════════════════════════════════════════
def run_phase0_jd_parsing(jd_text: str) -> dict:
    print("\n" + "=" * 55 + "\n📜 PHASE 0: JD INTELLIGENCE\n" + "=" * 55)

    jd_llm = LLM(
        model=QWEN_7B_PATH,
        quantization=QUANT_7B,
        max_model_len=4096,
        gpu_memory_utilization=0.70,
        swap_space=1,
        max_num_seqs=1,
        dtype="float16",
    )
    params = SamplingParams(temperature=0.0, max_tokens=1024)
    SYS = "You are a precise technical recruiter,you will be analyzing job description. Return valid JSON only, no markdown, no explanation."

    pass_prompts = {
        "explicit": (
            "Extract the explicit technical requirements from this job description as JSON. "
            "Keys: required_skills (array), required_experience_years (int), "
            "must_have_deployment_experience (bool), must_have_eval_framework_exp (bool), "
            "preferred_domain (string).\n\nJD:\n" + jd_text[:3000]
        ),
        "implicit": (
            "What does this JD IMPLICITLY require that is not stated directly? "
            "Return JSON with: implicit_skills (array), cultural_requirements (array), "
            "implicit_signals (array of brief strings describing unstated expectations).\n\nJD:\n"
            + jd_text[:3000]
        ),
        "anti": (
            "List all DISQUALIFYING backgrounds this JD mentions explicitly or implicitly. "
            "Return JSON with: disqualified_backgrounds (array), red_flag_titles (array), "
            "red_flag_company_types (array).\n\nJD:\n" + jd_text[:3000]
        ),
    }

    results = {}
    for pass_name, user_prompt in pass_prompts.items():
        prompt = (f"<|im_start|>system\n{SYS}\n<|im_end|>\n"
                  f"<|im_start|>user\n{user_prompt}\n<|im_end|>\n"
                  f"<|im_start|>assistant\n{{")
        raw = jd_llm.generate([prompt], params)[0].outputs[0].text.strip()
        raw = clean_json_output("{" + raw)
        try: results[pass_name] = json.loads(raw)
        except Exception: results[pass_name] = {"raw": raw}
        print(f"  ✓ {pass_name} pass")

    del jd_llm; flush()

    all_kw = set(JD_CORE_SKILLS)
    if isinstance(results.get("explicit"), dict):
        all_kw.update(s.lower() for s in results["explicit"].get("required_skills", []))
    if isinstance(results.get("implicit"), dict):
        all_kw.update(s.lower() for s in results["implicit"].get("implicit_skills", []))

    jd_context = {
        "explicit":            results.get("explicit", {}),
        "implicit":            results.get("implicit", {}),
        "anti":                results.get("anti", {}),
        "keywords_for_prompt": ", ".join(sorted(all_kw)[:30]),
        "full_jd_snippet":     jd_text[:1500],
    }
    print("✅ Phase 0 complete — jd_context ready.\n")
    return jd_context

# ══════════════════════════════════════════════════════════
# PHASE 1A: MASS EXTRACTION
# ══════════════════════════════════════════════════════════
def run_phase1a_mass_extraction(csv_input: str, jd_context: dict):
    print("=" * 55 + "\n⚡ PHASE 1A: MASS EXTRACTION\n" + "=" * 55)

    processed_ids  = set()
    header_written = False

    if os.path.exists(PHASE1A_CSV):
        try:
            existing_df   = pd.read_csv(PHASE1A_CSV, usecols=["candidate_id"])
            processed_ids = set(existing_df["candidate_id"].tolist())
            header_written = True
            print(f"🔄 Checkpoint Found: {len(processed_ids)} candidates already processed.")
        except Exception:
            os.remove(PHASE1A_CSV)

    if len(processed_ids) >= TARGET_ROWS:
        return

    extractor = LLM(
        model=QWEN_7B_PATH,
        quantization=QUANT_3B,
        max_model_len=3072,
        gpu_memory_utilization=0.75,
        swap_space=1,
        max_num_seqs=64,
        enable_prefix_caching=True,
        dtype="float16",
    )

    params = SamplingParams(temperature=0.0, max_tokens=300)
    jd_kw  = jd_context.get("keywords_for_prompt", "embeddings, retrieval, production, ranking")
    SYSTEM = (
        "You are a strict, deterministic data extraction engine. Your sole function is to map explicit, professional engineering facts from the candidate's profile into a predefined JSON structure.\n\n"
        "CRITICAL GUARDRAILS:\n"
        "1. ZERO INFERENCE: Do not assume, guess, or deduce any capabilities. If a skill or metric is not explicitly written in the text, it does not exist.\n"
        "2. NO HALLUCINATION: Extract only real, established technical AI/ML engineering skills directly mentioned. Do not invent frameworks.\n"
        "3. STRICT BOOLEANS: Set boolean fields to `true` ONLY if concrete, explicit professional evidence exists. If unsure, default to `false`.\n"
        "4. MISSING DATA: If information for a field is absent, output `[]`, `false`, `null`, or `0` depending on the schema type.\n\n"
        "HOBBYIST & LEARNER FILTERS (STRICT EXCLUSIONS):\n"
        "- IGNORE CONSUMER LLM USAGE: If a candidate mentions 'using ChatGPT', 'playing with Midjourney', 'prompt engineering', or generic AI usage, DO NOT extract these as AI skills. We are looking for engineers who *build* models, not consumers who talk to them.\n"
        "-IGNORE LEARNERS WHO ARE DOING ANY COURSE OR STUFF WE WANT PRACTICAL TALENT"
        "- BE AWARE OF GENRIC AI TERMINOLOGY WRITTEN BY CANDIDATE IGNORE IT IF NOT PROFFESIONALLY DEVLOPMENT CONCENTRATED"
        "- IGNORE LEARNER SIGNALS: If a candidate is merely 'learning AI', taking a basic tutorial, or describes themselves as an 'AI enthusiast', treat them as having NO professional AI skills. Their `ai_skills` must be left empty `[]`.\n\n"
        "FIELD DEFINITIONS:\n"
        "- ai_skills: [List exact engineering tools, architectures, algorithms, or vector DBs built/implemented (e.g., PyTorch, Qdrant, RAG, CNNs). Completely omit generic terms like 'AI', 'LLM', or 'ChatGPT'.]\n"
        "- ai skills are proffesional ai devlopment related domain not using ai should be listed.Realistic ai skills are real ai domain related stuff\n"
        "- deployment_evidence: [true ONLY if the text contains professional verbs showing they served, productionized, or shipped code to users.]\n"
        "- domain: [Classify as 'nlp_ir', 'cv_robotics', or 'general_ml' ONLY if they have concrete engineering skills. If they are a standard developer just 'learning' or using ChatGPT, you MUST classify them as 'software_only' or 'other'.]\n"
        "- coding_recency_years: [Calculate strictly from the most recent technical role provided.]\n\n"
        f"Target JSON Schema:\n{EXTRACTION_SCHEMA_STRING}\n\n"
        "OUTPUT REQUIREMENT:\n"
        "Output absolutely nothing except a single, valid JSON object. Do not include markdown blocks, explanations, or introductory text."
    )
    num_chunks = math.ceil(TARGET_ROWS / CHUNK_SIZE)

    for chunk_idx, chunk_df in enumerate(
        tqdm(pd.read_csv(csv_input, chunksize=CHUNK_SIZE, nrows=TARGET_ROWS), total=num_chunks, desc="1A chunks")
    ):
        chunk_df = chunk_df[~chunk_df["candidate_id"].isin(processed_ids)]
        if chunk_df.empty: continue

        blobs, prompts = [], []

        for _, row in chunk_df.iterrows():
            prof   = safe_eval(row.get("profile",        "{}"))
            career = safe_list(row.get("career_history", "[]"))
            skills = safe_list(row.get("skills",         "[]"))
            sigs   = safe_eval(row.get("redrob_signals", "{}"))

            desc_text   = " ".join(str(r.get("description", "") if isinstance(r, dict) else "").lower() for r in career)
            blob_skills = " ".join(s.get("name", "") for s in skills if isinstance(s, dict))
            blob_career = f"{prof.get('summary', '')} {desc_text}"[:1200]
            blob_edu    = str(safe_eval(row.get("education", "{}")))[:400]

            hp_risk, hp_flags                          = compute_honeypot_risk(career, prof, skills)
            beh_score, loc_m, notice_m, beh_flags, notice_days = compute_behavioral_composite(sigs, prof)
            rule_pen, rule_boost, rule_flags           = compute_rule_signals(career)
            rule_score = min(max(0.5 + rule_boost - rule_pen, 0.05), 1.0)

            all_flags = beh_flags + rule_flags + hp_flags

            blobs.append({
                "candidate_id":      row["candidate_id"],
                "point_id":          cid_to_point_id(row["candidate_id"]),
                "blob_skills":       blob_skills,
                "blob_career":       blob_career,
                "blob_education":    blob_edu,
                "behavioral_score":  round(beh_score,  4),
                "rule_score":        round(rule_score,  4),
                "loc_multiplier":    round(loc_m,       4),
                "notice_multiplier": round(notice_m,    4),
                "honeypot_risk":     round(hp_risk,     4),
                "notice_days":       notice_days,
                "red_flags":         " | ".join(all_flags) if all_flags else "",
                "composite_score":   0.0,
                "extraction_score":  0.0,
                "extracted_json":    "",
            })

            prompts.append(
                f"<|im_start|>system\n{SYSTEM}\n<|im_end|>\n"
                f"<|im_start|>user\nProfile:\n{blob_career}\n<|im_end|>\n"
                f"<|im_start|>assistant\n{{"
            )

        outputs = extractor.generate(prompts, params)

        for i, out in enumerate(outputs):
            raw_json  = clean_json_output("{" + out.outputs[0].text.strip())
            ext_score, ext_flags = score_from_extraction(raw_json)

            blobs[i]["extraction_score"] = round(ext_score, 4)
            blobs[i]["extracted_json"]   = raw_json
            if ext_flags:
                blobs[i]["red_flags"] = (blobs[i]["red_flags"] + " | " + " | ".join(ext_flags)).strip(" | ")

            blobs[i]["composite_score"] = round(
                (0.40 * ext_score + 0.35 * blobs[i]["rule_score"] + 0.25 * blobs[i]["behavioral_score"])
                * blobs[i]["loc_multiplier"] * blobs[i]["notice_multiplier"], 4
            )

        df_out = pd.DataFrame(blobs)
        df_out.to_csv(PHASE1A_CSV, mode="a", header=(not header_written), index=False)
        header_written = True

        del chunk_df, blobs, prompts, outputs, df_out
        gc.collect()

    del extractor; flush()
    print(f"✅ Phase 1A complete → {PHASE1A_CSV}\n")

# ══════════════════════════════════════════════════════════
# PHASE 1B: EVIDENCE REASONING
# ══════════════════════════════════════════════════════════

# BUG 2 FIX: "ddef" typo corrected to "def"
def build_multistrat_top1k(df: pd.DataFrame) -> pd.DataFrame:
    def safe_nlargest(df_sub, n, col):
        if col not in df_sub.columns or df_sub.empty:
            return pd.DataFrame(columns=df.columns)
        return df_sub.nlargest(n, col)

    seen = set()
    s1 = safe_nlargest(df, 550, "composite_score")
    seen.update(s1["candidate_id"].tolist())

    s2 = safe_nlargest(df[~df["candidate_id"].isin(seen)], 150, "behavioral_score")
    seen.update(s2["candidate_id"].tolist())

    def has_deployment_lang(text: str) -> bool:
        t = str(text).lower()
        return sum(kw in t for kw in DEPLOYMENT_KEYWORDS) >= 3

    remaining_3 = df[~df["candidate_id"].isin(seen)]
    if "blob_career" in remaining_3.columns:
        mask = remaining_3["blob_career"].apply(has_deployment_lang)
        s3   = safe_nlargest(remaining_3[mask], 150, "composite_score")
        seen.update(s3["candidate_id"].tolist())
    else:
        s3 = pd.DataFrame(columns=df.columns)

    remaining_4 = df[~df["candidate_id"].isin(seen)]
    if "composite_score" in remaining_4.columns:
        mid        = remaining_4["composite_score"].between(0.35, 0.65)
        notice_col = remaining_4.get("notice_days", pd.Series(60, index=remaining_4.index))
        s4         = safe_nlargest(remaining_4[mid & (notice_col <= 30)], 100, "composite_score")
    else:
        s4 = pd.DataFrame(columns=df.columns)

    combined = pd.concat([s1, s2, s3, s4]).drop_duplicates(subset="candidate_id").head(1000)
    print(f"  Strata pool selected: {len(combined)} candidates")
    return combined


def run_phase1b_evidence_reasoning(jd_context: dict) -> pd.DataFrame:
    print("=" * 55 + "\n🧠 PHASE 1B: EVIDENCE REASONING\n" + "=" * 55)

    df_1a  = pd.read_csv(PHASE1A_CSV)
    df_top = build_multistrat_top1k(df_1a)
    del df_1a; gc.collect()

    reasoner = LLM(
        model=QWEN_7B_PATH,
        quantization=QUANT_7B,
        max_model_len=3072,
        gpu_memory_utilization=0.70,
        swap_space=1,
        max_num_seqs=16,
        enable_prefix_caching=True,
        dtype="float16",
    )
    params = SamplingParams(temperature=0.1, max_tokens=450)

    anti     = jd_context.get("anti", {})
    implicit = jd_context.get("implicit", {})

    REASONING_SYS = (
        "You are a senior technical recruiter screening for a retrieval/search ML engineer role.\n"
        f"Job context: {jd_context.get('full_jd_snippet', '')[:600]}\n\n"
        f"Disqualifiers: {json.dumps(anti.get('disqualified_backgrounds', []))}\n"
        f"Implicit signals: {json.dumps(implicit.get('implicit_signals', []))}\n\n"
        "Task: find IMPLICIT evidence of fit that keyword matching misses.\n"
        "Also check: does the career description contain CONCRETE deployment evidence be very optimistic "
        "(words or semantically similar words like: served, shipped, production, endpoint, latency, QPS, A/B test, pipeline) or semainticaly similar words which depicts deployement skills of candidate?\n"
        "Return pure JSON — no markdown, no explanation:\n"
        '{"implicit_ai_evidence": "<string>", '
        '"has_deployment_evidence": true|false, '
        '"evidence_strength": "strong|moderate|weak|none", '
        '"key_concern": "<string or null>", '
        '"top_skill_signal": "<single most relevant skill or phrase from career text>"}'
    )

    prompts = [
        f"<|im_start|>system\n{REASONING_SYS}\n<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Skills blob: {str(row.get('blob_skills', ''))[:400]}\n"
        f"Career:\n{str(row.get('blob_career', ''))[:1000]}\n"
        f"<|im_end|>\n<|im_start|>assistant\n{{"
        for _, row in df_top.iterrows()
    ]

    outputs   = reasoner.generate(prompts, params)
    enriched  = []
    boost_map = {"strong": 0.20, "moderate": 0.10, "weak": 0.03, "none": 0.0}

    for i, out in enumerate(outputs):
        raw = clean_json_output("{" + out.outputs[0].text.strip())
        try:
            parsed = json_repair.loads(raw)
            if not isinstance(parsed, dict): parsed = {}
        except Exception:
            parsed = {}

        strength   = parsed.get("evidence_strength", "weak")
        boost      = boost_map.get(strength, 0.0)
        base_score = float(df_top.iloc[i].get("composite_score", 0.0))
        hp_risk    = float(df_top.iloc[i].get("honeypot_risk",   0.0))

        enriched_score = 0.10 if hp_risk >= 0.80 else min(base_score + boost, 1.0)
        deployment_found = bool(parsed.get("has_deployment_evidence", False))

        enriched.append({
            "implicit_evidence":        parsed.get("implicit_ai_evidence", ""),
            "evidence_strength":        strength,
            "key_concern":              parsed.get("key_concern"),
            "top_skill_signal":         parsed.get("top_skill_signal", ""),
            "deployment_confirmed_1b":  deployment_found,
            "enriched_composite_score": round(enriched_score, 4),
        })

    df_top = df_top.reset_index(drop=True)
    new_cols = pd.DataFrame(enriched)
    for col in new_cols.columns:
        df_top[col] = new_cols[col]

    df_top.to_csv(PHASE1B_CSV, index=False)
    del reasoner; flush()
    print(f"✅ Phase 1B complete → {PHASE1B_CSV}\n")
    return df_top

# ══════════════════════════════════════════════════════════
# PHASE 1C: EMBEDDING + QDRANT INGEST
# ══════════════════════════════════════════════════════════
def run_phase1c_full_vector_ingest() -> QdrantClient:
    print("=" * 55 + "\n🧬 PHASE 1C: BGE-M3 ENCODING\n" + "=" * 55)
    client = QdrantClient(path=QDRANT_PATH)

    # BUG 3 FIX: removed the misplaced set_payload block that appeared before
    # create_collection; collections must exist before any upsert/patch can happen.
    for aspect in ASPECT_WEIGHTS:
        try: client.delete_collection(f"candidates_{aspect}")
        except Exception: pass
        client.create_collection(
            f"candidates_{aspect}",
            vectors_config=VectorParams(
                size=1024,
                distance=Distance.COSINE,
                on_disk=True  # <--- Forces vectors out of RAM
            ),
            hnsw_config=HnswConfigDiff(
                on_disk=True  # <--- Forces search index out of RAM
            )
        )

    bge = SentenceTransformer(BGE_M3_PATH)
    bge.max_seq_length = 512
    num_chunks = math.ceil(TARGET_ROWS / CHUNK_SIZE)

    for _, chunk_df in enumerate(tqdm(pd.read_csv(PHASE1A_CSV, chunksize=CHUNK_SIZE), total=num_chunks)):
        for aspect in ASPECT_WEIGHTS:
            col   = f"blob_{aspect}"
            texts = chunk_df[col].astype(str).fillna("").tolist() if col in chunk_df.columns else [""] * len(chunk_df)
            vecs  = bge.encode(texts, batch_size=128)

            points = [
                PointStruct(
                    id=int(row["point_id"]),
                    vector=vecs[j].tolist(),
                    payload={
                        "candidate_id":      str(row["candidate_id"]),
                        "composite_score":   float(row.get("composite_score",  0.0)),
                        "behavioral_score":  float(row.get("behavioral_score", 0.5)),
                        "extraction_score":  float(row.get("extraction_score", 0.3)),
                        "honeypot_risk":     float(row.get("honeypot_risk",    0.0)),
                        "red_flags":         str(row.get("red_flags", "")),
                        "notice_days":       int(row.get("notice_days", 60)),
                        "loc_multiplier":    float(row.get("loc_multiplier",    1.0)),
                        "notice_multiplier": float(row.get("notice_multiplier", 1.0)),
                        "blob_career":       str(row.get("blob_career", "")),
                        "blob_skills":       str(row.get("blob_skills", "")),
                        "implicit_evidence":     "",
                        "key_concern":           None,
                        "top_skill_signal":      "",
                        "deployment_confirmed":  False,
                    }
                ) for j, (_, row) in enumerate(chunk_df.iterrows())
            ]
            client.upsert(collection_name=f"candidates_{aspect}", points=points)
        del chunk_df; gc.collect()

    # BUG 4 FIX: outer loop was "for aspect in ASPECT_WEIGHTS" but the inner
    # set_payload was at the same indent level — making it run once per aspect
    # instead of once per candidate.  Correct structure: iterate candidates in
    # the outer loop, aspects in the inner loop.
    df_1b = pd.read_csv(PHASE1B_CSV)
    for _, row in df_1b.iterrows():
        pid = cid_to_point_id(row["candidate_id"])
        patch = {
            "composite_score":      float(row.get("enriched_composite_score", row.get("composite_score", 0.0))),
            "implicit_evidence":    str(row.get("implicit_evidence", "")),
            "key_concern":          row.get("key_concern"),
            "top_skill_signal":     str(row.get("top_skill_signal", "")),
            "deployment_confirmed": bool(row.get("deployment_confirmed_1b", False)),
        }
        for aspect in ASPECT_WEIGHTS:  # inner loop — correct
            try:
                client.set_payload(
                    collection_name=f"candidates_{aspect}",
                    payload=patch,
                    points=PointIdsList(points=[pid])
                )
            except Exception:
                pass

    del bge; flush()
    return client

# ══════════════════════════════════════════════════════════
# PHASE 1D: NATURAL LANGUAGE REASONING
# ══════════════════════════════════════════════════════════
def run_phase1d_naturalize_cot(top_100: list) -> list:
    """Unchanged from previous fix — included here for completeness."""
    print("\n  🖊  Phase 1D: naturalizing CoT for top-100...")
    naturalizer = LLM(
        model=QWEN_7B_PATH,
        quantization=QUANT_7B,
        max_model_len=768,
        gpu_memory_utilization=0.70,
        swap_space=1,
        max_num_seqs=64,
        dtype="float16",
    )
    params = SamplingParams(temperature=0.25, max_tokens=120)

    prompts = []
    for c in top_100:
        skill_signal = str(c.get("top_skill_signal", "")).strip()
        implicit_ev  = str(c.get("implicit_evidence", "")).strip()
        red_flags    = str(c.get("red_flags", "none")).strip(" |")
        key_concern  = str(c.get("key_concern") or "").strip()
        blob_skills  = str(c.get("blob_skills", ""))[:200]

        positive = skill_signal or implicit_ev or blob_skills or "general ML background"
        concern  = key_concern or red_flags or "no major concerns identified"

        prompts.append(
            f"<|im_start|>system\n"
            f"You are a technical recruiter writing a brief candidate note. "
            f"Write exactly TWO sentences. "
            f"Sentence 1: state the strongest technical signal this candidate shows (use the positive signal). "
            f"Sentence 2: state the main risk or gap (use the concern). "
            f"Be specific, not generic. Do NOT mention scores or numbers. "
            f"Do NOT start with 'The candidate'. Start with an active phrase about their work.\n"
            f"<|im_end|>\n"
            f"<|im_start|>user\n"
            f"Positive signal: {positive[:300]}\n"
            f"Concern: {concern[:200]}\n"
            f"<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

    outputs = naturalizer.generate(prompts, params)
    cots    = [o.outputs[0].text.strip() for o in outputs]
    del naturalizer; flush()
    return cots

# ══════════════════════════════════════════════════════════
# PHASE 2 & 3: ENSEMBLE RANKING
# ══════════════════════════════════════════════════════════
def run_phase2_3_online_ranking(client: QdrantClient, jd_text: str):
    print("=" * 55 + "\n🎯 PHASE 2+3: ONLINE CASCADE\n" + "=" * 55)

    bge_q = SentenceTransformer(BGE_M3_PATH)
    bge_q.max_seq_length = 1500
    jd_vec = bge_q.encode([jd_text[:1500]])[0].tolist()
    del bge_q; flush()


    merged = {}
    for aspect, weight in ASPECT_WEIGHTS.items():
        hits = client.query_points(
            collection_name=f"candidates_{aspect}", query=jd_vec, limit=TOP_QDRANT_N
        ).points
        for h in hits:
            cid = h.payload.get("candidate_id")
            if cid not in merged:
                merged[cid] = {"payload": h.payload, "semantic_score": 0.0}
            merged[cid]["semantic_score"] += h.score * weight

    candidates = [
        v for v in merged.values()
        if float(v["payload"].get("honeypot_risk", 0)) < HP_THRESHOLD
    ]
    candidates.sort(key=lambda x: x["semantic_score"], reverse=True)
    candidates = candidates[:RERANK_N]
    print(f"  Candidates after HP filter: {len(candidates)}")


    reranker_scores = np.full(len(candidates), 0.5)
    reranker_loaded = False

    CROSS_ENCODER_MODELS = [
        "BAAI/bge-reranker-base",
        "cross-encoder/ms-marco-MiniLM-L-6-v2",
    ]

    for model_name in ["BAAI/bge-reranker-base", "cross-encoder/ms-marco-MiniLM-L-6-v2"]:
        try:
            print(f"  Loading reranker: {model_name}")
            reranker = CrossEncoder(
                model_name, max_length=512,
                device="cuda" if torch.cuda.is_available() else "cpu"
            )
            pairs = [
                (
                    jd_text[:512],
                    (c["payload"].get("blob_skills", "") + " " +
                     c["payload"].get("blob_career", ""))[:512]
                )
                for c in candidates
            ]
            raw   = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
            arr   = np.array(raw, dtype=float)
            lo, hi = arr.min(), arr.max()
            reranker_scores = (arr - lo) / (hi - lo) if hi > lo else np.full(len(candidates), 0.5)
            del reranker; flush()
            reranker_loaded = True
            print(f"  ✅ Reranker scores: {reranker_scores.min():.3f} – {reranker_scores.max():.3f}")
            break
        except Exception as e:
            print(f"  ⚠️  {model_name} failed: {e}")

    final = []
    for i, cand in enumerate(candidates):
        p = cand["payload"]

        sem_score  = float(cand["semantic_score"])
        rer_score  = float(reranker_scores[i])
        comp_score = float(p.get("composite_score",  0.30))
        beh_score  = float(p.get("behavioral_score", 0.50))
        ext_score  = float(p.get("extraction_score", 0.30))

        if reranker_loaded:
            raw_ensemble = (
                0.25 * sem_score
              + 0.35 * rer_score
              + 0.20 * comp_score   # carries loc × notice — NOT multiplied again
              + 0.12 * beh_score
              + 0.08 * ext_score
            )
        else:
            # reranker dead: redistribute its weight
            raw_ensemble = (
                0.25 * sem_score
              + 0.35 * comp_score
              + 0.10 * beh_score
              + 0.30 * ext_score
            )

        # ── soft location/notice nudge (replaces the hard double-multiply) ──
        loc_m    = float(p.get("loc_multiplier",    1.0))
        notice_m = float(p.get("notice_multiplier", 1.0))

        # Nudge: scale 0.20–1.00 loc range → −0.05 to +0.02 additive adjustment
        loc_nudge    = (loc_m    - 0.80) * 0.10   # -0.060 at loc=0.20, +0.020 at loc=1.00
        notice_nudge = (notice_m - 0.80) * 0.05   # small adjustment for notice period

        final_score = round(
            max(0.0, min(raw_ensemble + loc_nudge + notice_nudge, 1.0)), 4
        )
        final.append({**p, "ensemble_score": final_score})

    final.sort(key=lambda x: x["ensemble_score"], reverse=True)
    top_100 = final[:SUBMISSION_N]


    natural_cots = run_phase1d_naturalize_cot(top_100)

    scores = [c["ensemble_score"] for c in top_100]
    print(f"\n  Pre-CoT score check → max: {max(scores):.4f}  median: {sorted(scores)[50]:.4f}  min: {min(scores):.4f}")
    if max(scores) < 0.15:
        print("  ⚠️  WARNING: scores still very low — check Phase 1A composite_score values in payload")

    natural_cots = run_phase1d_naturalize_cot(top_100)


    rows = []
    for rank, (cand, cot) in enumerate(zip(top_100, natural_cots), start=1):

        raw_concern = cand.get("key_concern")
        concern = (
            "" if (not raw_concern or str(raw_concern).lower() in ("nan", "none", "null"))
            else str(raw_concern).strip()
        )

        raw_flags = cand.get("red_flags", "")
        red_flags = (
            "" if (not raw_flags or str(raw_flags).lower() in ("nan", "none"))
            else str(raw_flags).strip(" |")
        )

        # Remove no_prod_deployment if Phase 1B confirmed actual deployment
        if cand.get("deployment_confirmed") and "no_prod_deployment" in red_flags:
            red_flags = " | ".join(
                f for f in red_flags.split(" | ")
                if f.strip() != "no_prod_deployment"
            )

        combined    = [f for f in [red_flags, concern] if f]
        final_flags = " | ".join(combined) if combined else "None"


        rows.append({
            "rank":         rank,
            "candidate_id": cand["candidate_id"],
            "score":        cand["ensemble_score"],
            "tier":         tier_from_score(cand["ensemble_score"]),
            "reasoning":    cot,
            "red_flags":    final_flags,
        })

    df_out = pd.DataFrame(rows)
    df_out.to_csv(FINAL_CSV, index=False)
    print(f"\n🎉 Submission CSV → {FINAL_CSV}")
    _print_score_summary(df_out)
    return df_out


# BUG 5 FIX: was indented inside run_phase2_3_online_ranking (after its return),
# making it unreachable dead code.  Moved to module level.
def _print_score_summary(df: pd.DataFrame):
    from collections import Counter
    print("\n📊 Score distribution:")
    print(f"  Max:    {df['score'].max():.4f}")
    print(f"  Median: {df['score'].median():.4f}")
    print(f"  Min:    {df['score'].min():.4f}")
    print(f"  Std:    {df['score'].std():.4f}")
    print("\n📊 Tier breakdown:")
    print(df["tier"].value_counts().sort_index().to_string())
    print("\n📊 Top-10 red flag frequencies:")
    all_flags = []
    for flags_str in df["red_flags"]:
        if flags_str and str(flags_str).lower() != "none":
            all_flags.extend([f.strip() for f in str(flags_str).split("|") if f.strip()])
    for flag, count in Counter(all_flags).most_common(10):
        print(f"  {flag}: {count}")


# ══════════════════════════════════════════════════════════
# ENTRYPOINT
# ══════════════════════════════════════════════════════════
if __name__ == "__main__":
    import docx

    setup_local_models()

    # Load Job Description
    doc = docx.Document(JD_PATH)
    jd_content = "\n".join(
        p.text for p in doc.paragraphs if p.text.strip()
    )

    # -------------------------
    # Phase 0 (with cache)
    # -------------------------
    jd_context_path = os.path.join(DRIVE_DIR, "jd_context.json")

    if os.path.exists(jd_context_path):
        with open(jd_context_path, "r") as f:
            jd_context = json.load(f)
        print("✅ Loaded jd_context from cache")
    else:
        jd_context = run_phase0_jd_parsing(jd_content)
        with open(jd_context_path, "w") as f:
            json.dump(jd_context, f)
        print("✅ Saved jd_context to cache")

    # -------------------------
    # Phase 1A
    # Always call it.
    # It will resume automatically if checkpoint exists.
    # -------------------------
    run_phase1a_mass_extraction(DATA_PATH, jd_context)

    # -------------------------
    # Phase 1B
    # -------------------------
    run_phase1b_evidence_reasoning(jd_context)

    # -------------------------
    # Phase 1C
    # -------------------------
    qdrant_client = run_phase1c_full_vector_ingest()

    # -------------------------
    # Phase 2 & 3
    # -------------------------
    run_phase2_3_online_ranking(qdrant_client, jd_content)

/usr/local/lib/python3.12/dist-packages/vllm/connections.py:8: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from vllm.version import __version__ as VLLM_VERSION


🔥 RUNNING IN PRODUCTION MODE (100,000 Candidates)
📥 CHECKING LOCAL MODEL STORAGE
✅ Models ready.

✅ Loaded jd_context from cache
⚡ PHASE 1A: MASS EXTRACTION
🔄 Checkpoint Found: 100000 candidates already processed.
🧠 PHASE 1B: EVIDENCE REASONING
  Strata pool selected: 950 candidates
WARNING 06-29 06:29:23 config.py:306] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 06-29 06:29:23 llm_engine.py:237] Initializing an LLM engine (vdev) with config: model='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=3072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disabl

/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 06-29 06:29:26 model_runner.py:1060] Starting to load model /content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ...
INFO 06-29 06:29:26 selector.py:224] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 06-29 06:29:26 selector.py:115] Using XFormers backend.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


...


  Pre-CoT score check → max: 0.7265  median: 0.2002  min: 0.1521

  🖊  Phase 1D: naturalizing CoT for top-100...
WARNING 06-29 09:54:44 config.py:306] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 06-29 09:54:44 llm_engine.py:237] Initializing an LLM engine (vdev) with config: model='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(gui

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 06-29 09:55:14 model_runner.py:1071] Loading model weights took 5.1958 GB
INFO 06-29 09:55:16 gpu_executor.py:122] # GPU blocks: 4074, # CPU blocks: 1170
INFO 06-29 09:55:16 gpu_executor.py:126] Maximum concurrency for 768 tokens per request: 84.88x
INFO 06-29 09:55:16 model_runner.py:1402] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-29 09:55:16 model_runner.py:1406] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 06-29 09:55:31 model_runner.py:1530] Graph capturing finished in 15 secs.


Processed prompts: 100%|██████████| 100/100 [00:32<00:00,  3.12it/s, est. speed input: 381.87 toks/s, output: 134.47 toks/s]



🎉 Submission CSV → /content/drive/MyDrive/redrob_ai/team_cuda,coffee,code_submission004.csv

📊 Score distribution:
  Max:    0.7265
  Median: 0.1983
  Min:    0.1521
  Std:    0.1528

📊 Tier breakdown:
tier
T1     6
T2     8
T3     6
T4     1
T5    79

📊 Top-10 red flag frequencies:
  outside_india_no_relocation: 48
  no_public_code: 35
  inactive_6m+: 25
  low_response_rate: 16
  tenure_yoe_mismatch: 2
  no_deployment_evidence: 1


In [ ]:
!zip -r /content/qdrant_backup.zip /content/qdrant_storage

  adding: content/qdrant_storage/ (stored 0%)
  adding: content/qdrant_storage/.lock (stored 0%)
  adding: content/qdrant_storage/meta.json (deflated 83%)
  adding: content/qdrant_storage/collection/ (stored 0%)
  adding: content/qdrant_storage/collection/candidates_education/ (stored 0%)
  adding: content/qdrant_storage/collection/candidates_education/storage.sqlite (deflated 56%)
  adding: content/qdrant_storage/collection/candidates_skills/ (stored 0%)
  adding: content/qdrant_storage/collection/candidates_skills/storage.sqlite (deflated 56%)
  adding: content/qdrant_storage/collection/candidates_career/ (stored 0%)
  adding: content/qdrant_storage/collection/candidates_career/storage.sqlite (deflated 56%)


In [ ]:
!cp /content/qdrant_backup.zip /content/drive/MyDrive/redrob_ai/